QR integrated with ocr & face_recognition

In [4]:
# Register - Can handle rotate better

import cv2
import pytesseract
import face_recognition
import re
import os
import csv
import numpy as np
from PIL import Image, ImageTk
import tkinter as tk
from tkinter import messagebox, filedialog
from tkinter import ttk
import math
import time

#YF 
import qrcode
import smtplib
import socket
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email import encoders
from datetime import datetime, timedelta
import random

# Configure tesseract path
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# Email configuration
SENDER_EMAIL = "paradisebookstore888@gmail.com"
SENDER_PASSWORD = " iczp fdcl yzyr xtfs"  # Use App Password if Gmail

# Track failed attempts & lock status
failed_attempts = {}  # { email: {"count": int, "locked_until": datetime } }

# Global current image
current_image = None
verification_codes = {}  # Store verification codes temporarily

# ----------------- ENHANCED AUTO-CROP FUNCTION -----------------
def auto_crop_id_card(image):
    """
    Enhanced auto-crop function that detects 4 corners, removes background,
    corrects perspective, handles rotation, and standardizes to template format.
    """
    # Convert PIL image to OpenCV format
    img_array = np.array(image)
    img_array = cv2.cvtColor(img_array, cv2.COLOR_RGB2BGR)
    
    # Step 1: Try contour-based detection (primary for background removal)
    contoured = detect_id_card_contour(img_array)
    
    if contoured is not None:
        result = contoured
    else:
        # Fallback 1: Template-based matching with rotations
        print("Contour detection failed, trying template-based matching...")
        result = template_based_crop(img_array)
        
        if result is None:
            # Fallback 2: Edge-based cropping
            print("Template matching failed, trying edge-based cropping...")
            result = edge_based_crop(img_array)
    
    if result is None:
        print("All cropping methods failed, returning original image.")
        return image
    
    # Step 2: Correct orientation if needed
    result = correct_orientation(result)
    
    # Step 3: Post-process and standardize size
    final_result = post_process_cropped_image(result)
    final_result = resize_to_standard_size(final_result)
    
    # Convert back to PIL image
    result_pil = Image.fromarray(cv2.cvtColor(final_result, cv2.COLOR_BGR2RGB))
    
    return result_pil

def correct_orientation(image):
    """
    Detect if the image is upside down or rotated and correct it.
    Tries 0° and 180° first (common after perspective), then 90°/270° if needed.
    Uses quick OCR to check for student ID pattern.
    """
    rotation_angles = [0, 180, 90, 270]  # Prioritize likely corrections
    best_image = None
    best_confidence = 0
    
    for angle in rotation_angles:
        rotated = rotate_image(image, angle)
        gray = cv2.cvtColor(rotated, cv2.COLOR_BGR2GRAY)
        thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
        
        # Quick OCR config for detection only (focus on ID pattern)
        custom_config = r'--oem 3 --psm 6 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
        text = pytesseract.image_to_string(thresh, config=custom_config)
        
        # Look for student ID pattern (e.g., 24WMR09148)
        id_match = re.search(r'\d{2}[A-Z]{3}\d{5}', text.upper())
        if id_match:
            # If ID found, assume correct orientation (higher confidence if more text)
            confidence = len(text.strip())
            if confidence > best_confidence:
                best_confidence = confidence
                best_image = rotated
    
    return best_image if best_image is not None else image

def detect_id_card_contour(image):
    """
    Detect ID card using contour detection with color-based pre-processing
    and perspective correction, optimized for background and tilt.
    """
    original = image.copy()
    
    # Resize image for processing
    height, width = image.shape[:2]
    scale_factor = max(0.5, min(1.0, 800 / width))
    if scale_factor < 1.0:
        new_width = int(width * scale_factor)
        new_height = int(height * scale_factor)
        image = cv2.resize(image, (new_width, new_height))
    
    # Step 1: Color-based pre-processing to isolate the card
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    # Define range for white background (adjust based on card color)
    lower_white = np.array([0, 0, 200])  # Lower bound for white
    upper_white = np.array([180, 30, 255])  # Upper bound for white
    mask = cv2.inRange(hsv, lower_white, upper_white)
    
    # Refine mask with morphological operations
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    mask = cv2.erode(mask, kernel, iterations=1)
    mask = cv2.dilate(mask, kernel, iterations=1)
    
    # Apply mask to original image
    masked_image = cv2.bitwise_and(image, image, mask=mask)
    
    # Step 2: Convert to grayscale and enhance contrast
    gray = cv2.cvtColor(masked_image, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    
    # Step 3: Edge detection with adjusted parameters
    edges = cv2.Canny(gray, 30, 120, apertureSize=3, L2gradient=True)  # Lowered thresholds
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
    dilated = cv2.dilate(edges, kernel, iterations=2)  # Increased dilation
    
    # Step 4: Find contours
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        print("No contours detected after pre-processing. Check lighting or card visibility.")
        return None
    
    # Step 5: Find the best card contour
    card_contour = find_card_contour(contours, image.shape)
    
    if card_contour is not None:
        # Scale contour back to original image size
        if scale_factor < 1.0:
            card_contour = (card_contour / scale_factor).astype(np.int32)
        
        # Apply perspective correction
        warped = perspective_correction(original, card_contour)
        print(f"Successfully warped card with contour area: {cv2.contourArea(card_contour)}")
        return warped
    
    print("No suitable card contour found after all attempts.")
    return None

def find_card_contour(contours, image_shape):
    """
    Find the contour that best represents an ID card, with improved filtering
    and background rejection.
    """
    h, w = image_shape[:2]
    min_area = (w * h) * 0.03  # Further lowered to 3% for small cards
    max_area = (w * h) * 0.95
    best_contour = None
    best_score = 0
    
    for contour in contours:
        area = cv2.contourArea(contour)
        if area < min_area or area > max_area:
            continue
        
        # Approximate contour to polygon
        epsilon = 0.03 * cv2.arcLength(contour, True)  # Slightly larger epsilon
        approx = cv2.approxPolyDP(contour, epsilon, True)
        
        # Look for quadrilateral shapes
        if 4 <= len(approx) <= 8:  # Allow more flexibility
            x, y, w_rect, h_rect = cv2.boundingRect(approx)
            aspect_ratio = w_rect / h_rect if h_rect > 0 else 1.0
            
            # Check for reasonable aspect ratio and size
            if 1.1 < aspect_ratio < 2.5:  # Widened range
                contour_area = cv2.contourArea(approx)
                bounding_area = w_rect * h_rect
                extent = contour_area / bounding_area if bounding_area > 0 else 0
                
                # Score based on area, aspect ratio, and rectangularity
                score = area * extent
                if 1.4 <= aspect_ratio <= 1.7:  # Boost for ideal ratio
                    score *= 1.2
                
                # Prioritize contours with higher solidity and central position
                center_x = x + w_rect / 2
                center_y = y + h_rect / 2
                centrality = 1 - abs(center_x / w - 0.5) - abs(center_y / h - 0.5)
                score *= centrality * 1.5  # Favor centrally located contours
                
                if score > best_score:
                    best_score = score
                    best_contour = approx.reshape(-1, 2)
    
    return best_contour

def perspective_correction(image, contour):
    """
    Apply perspective correction with validation for better alignment.
    """
    # Order points: top-left, top-right, bottom-right, bottom-left
    rect = order_points(contour)
    if rect is None or np.any(np.isnan(rect)):
        print("Invalid contour points for perspective correction.")
        return None
    
    # Calculate the width and height of the new image
    (tl, tr, br, bl) = rect
    widthA = np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
    widthB = np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))
    maxWidth = max(int(widthA), int(widthB))
    
    heightA = np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
    heightB = np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))
    maxHeight = max(int(heightA), int(heightB))
    
    # Standard ID card aspect ratio (approximately 1.585)
    standard_ratio = 1.585
    if maxWidth / maxHeight > standard_ratio * 1.2:  # Allow 20% tolerance
        maxHeight = int(maxWidth / standard_ratio)
    elif maxHeight / maxWidth > standard_ratio * 1.2:
        maxWidth = int(maxHeight / standard_ratio)
    
    # Define destination points
    dst = np.array([
        [0, 0],
        [maxWidth - 1, 0],
        [maxWidth - 1, maxHeight - 1],
        [0, maxHeight - 1]], dtype="float32")
    
    # Calculate perspective transform
    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (maxWidth, maxHeight))
    
    return warped

def order_points(pts):
    """
    Order points in the order: top-left, top-right, bottom-right, bottom-left.
    Handle cases where points might be noisy.
    """
    if len(pts) < 4:
        return None
    
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1)
    
    # Handle potential outliers by sorting and taking top 4
    indices = np.argsort(s)
    rect[0] = pts[indices[0]]  # Top-left (smallest sum)
    rect[2] = pts[indices[-1]]  # Bottom-right (largest sum)
    
    diff_indices = np.argsort(diff[:, 0])
    rect[1] = pts[diff_indices[0]]  # Top-right (smallest diff)
    rect[3] = pts[diff_indices[-1]]  # Bottom-left (largest diff)
    
    return rect

def template_based_crop(img_array):
    """
    Fallback method using template matching (existing implementation)
    """
    # Load reference template
    template = cv2.imread('template_id.jpg')
    if template is None:
        print("Template image not found.")
        return None
    
    # Preprocess both images
    img_processed = preprocess_for_matching(img_array)
    template_processed = preprocess_for_matching(template)
    
    # Try different rotation angles
    rotation_angles = [0, 90, 180, 270]
    best_result = None
    best_score = 0
    
    for angle in rotation_angles:
        # Rotate the input image
        rotated_img = rotate_image(img_array, angle)
        rotated_processed = preprocess_for_matching(rotated_img)
        
        # Try to match with template
        result, score = match_and_align(rotated_processed, template_processed, rotated_img, template)
        
        if score > best_score:
            best_score = score
            best_result = result
    
    return best_result

def preprocess_for_matching(image):
    """
    Preprocess image for better feature matching
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # Enhance contrast
    enhanced = cv2.convertScaleAbs(gray, alpha=1.3, beta=20)
    # Apply Gaussian blur to reduce noise
    blurred = cv2.GaussianBlur(enhanced, (3, 3), 0)
    return blurred

def rotate_image(image, angle):
    """
    Rotate image by specified angle
    """
    if angle == 0:
        return image
    elif angle == 90:
        return cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
    elif angle == 180:
        return cv2.rotate(image, cv2.ROTATE_180)
    elif angle == 270:
        return cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)
    else:
        # For any other angle, use rotation matrix
        h, w = image.shape[:2]
        center = (w // 2, h // 2)
        rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
        return cv2.warpAffine(image, rotation_matrix, (w, h))

def match_and_align(processed_img, processed_template, original_img, original_template):
    """
    Match features and align the image
    """
    # Initialize ORB detector with more features
    orb = cv2.ORB_create(nfeatures=2000, scaleFactor=1.2, nlevels=8)
    
    # Find keypoints and descriptors
    kp1, des1 = orb.detectAndCompute(processed_template, None)
    kp2, des2 = orb.detectAndCompute(processed_img, None)
    
    if des1 is None or des2 is None:
        return None, 0
    
    # Match features using FLANN matcher for better results
    FLANN_INDEX_LSH = 6
    index_params = dict(algorithm=FLANN_INDEX_LSH, table_number=6, key_size=12, multi_probe_level=1)
    search_params = dict(checks=50)
    flann = cv2.FlannBasedMatcher(index_params, search_params)
    
    try:
        matches = flann.knnMatch(des1, des2, k=2)
    except:
        # Fallback to BF matcher
        bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
        raw_matches = bf.match(des1, des2)
        matches = [[m] for m in sorted(raw_matches, key=lambda x: x.distance)]
    
    # Filter good matches using Lowe's ratio test
    good_matches = []
    for match_pair in matches:
        if len(match_pair) == 2:
            m, n = match_pair
            if m.distance < 0.7 * n.distance:
                good_matches.append(m)
        elif len(match_pair) == 1:
            good_matches.append(match_pair[0])
    
    print(f"Found {len(good_matches)} good matches")
    
    if len(good_matches) < 10:
        return None, 0
    
    # Select best matches
    good_matches = sorted(good_matches, key=lambda x: x.distance)[:min(50, len(good_matches))]
    
    # Extract matching points
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    
    # Find homography
    homography, mask = cv2.findHomography(dst_pts, src_pts, 
                                        cv2.RANSAC, 
                                        ransacReprojThreshold=5.0,
                                        confidence=0.99)
    
    if homography is None:
        return None, 0
    
    # Calculate match score based on inliers
    inliers = np.sum(mask)
    score = inliers / len(good_matches)
    
    # Warp the image
    h, w = original_template.shape[:2]
    aligned_img = cv2.warpPerspective(original_img, homography, (w, h))
    
    return aligned_img, score

def edge_based_crop(image):
    """
    Fallback method using edge detection to find ID card boundaries
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Apply Gaussian blur
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Edge detection
    edges = cv2.Canny(blurred, 50, 150, apertureSize=3)
    
    # Find contours
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Find the largest rectangular contour
    largest_area = 0
    best_contour = None
    
    for contour in contours:
        # Approximate contour to polygon
        epsilon = 0.02 * cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, epsilon, True)
        
        # Check if it's roughly rectangular (4 corners)
        if len(approx) >= 4:
            area = cv2.contourArea(contour)
            if area > largest_area and area > 10000:  # Minimum area threshold
                largest_area = area
                best_contour = approx
    
    if best_contour is None:
        return None
    
    # Get bounding rectangle
    x, y, w, h = cv2.boundingRect(best_contour)
    
    # Add some padding
    padding = 10
    x = max(0, x - padding)
    y = max(0, y - padding)
    w = min(image.shape[1] - x, w + 2 * padding)
    h = min(image.shape[0] - y, h + 2 * padding)
    
    # Crop the image
    cropped = image[y:y+h, x:x+w]
    
    # Resize to standard ID card dimensions (maintaining aspect ratio)
    return resize_to_standard_size(cropped)

def resize_to_standard_size(image):
    """
    Resize image to standard ID card size while maintaining aspect ratio
    """
    # Standard ID card dimensions (adjust as needed)
    target_width = 800
    target_height = 504  # Approximately 1.585 aspect ratio
    
    h, w = image.shape[:2]
    
    # Calculate scaling factor
    scale_w = target_width / w
    scale_h = target_height / h
    scale = min(scale_w, scale_h)
    
    # Calculate new dimensions
    new_w = int(w * scale)
    new_h = int(h * scale)
    
    # Resize image
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LANCZOS4)
    
    # Create canvas with target size
    canvas = np.zeros((target_height, target_width, 3), dtype=np.uint8)
    canvas.fill(255)  # White background
    
    # Center the image on canvas
    start_x = (target_width - new_w) // 2
    start_y = (target_height - new_h) // 2
    
    canvas[start_y:start_y+new_h, start_x:start_x+new_w] = resized
    
    return canvas

def post_process_cropped_image(image):
    """
    Post-process the cropped image for better OCR results
    """
    # Convert to grayscale
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image.copy()
    
    # Enhance contrast and brightness
    enhanced = cv2.convertScaleAbs(gray, alpha=1.2, beta=15)
    
    # Apply slight Gaussian blur to reduce noise
    denoised = cv2.GaussianBlur(enhanced, (3, 3), 0)
    
    # Convert back to BGR for consistency
    result = cv2.cvtColor(denoised, cv2.COLOR_GRAY2BGR)
    
    return result

# ----------------- OCR & ID CLEANUP -----------------
def clean_student_id(ocr_id):
    return (
        ocr_id.upper()
        .replace("O9", "09")
        .replace("O", "0", 1)
        .replace("I", "1")
        .replace("S", "5")
    )
                     
def calibrate_roi_coordinates():
    """
    Helper function to manually calibrate ROI coordinates
    Run this once to fine-tune the extraction region
    """
    template_path = 'template_id.jpg'
    if not os.path.exists(template_path):
        print("Template image not found!")
        return
    
    template = cv2.imread(template_path)
    h, w = template.shape[:2]
    
    # Current ROI coordinates
    roi_x = int(w * 0.52)
    roi_y = int(h * 0.43)
    roi_width = int(w * 0.46)
    roi_height = int(h * 0.25)
    
    # Draw ROI rectangle on template
    template_copy = template.copy()
    cv2.rectangle(template_copy, (roi_x, roi_y), (roi_x + roi_width, roi_y + roi_height), (0, 255, 0), 2)
    cv2.putText(template_copy, "Name & ID Region", (roi_x, roi_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    # Display the template with ROI
    cv2.imshow("ROI Calibration - Press any key to close", template_copy)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    
    print(f"Current ROI coordinates:")
    print(f"X: {roi_x} ({roi_x/w*100:.1f}% of width)")
    print(f"Y: {roi_y} ({roi_y/h*100:.1f}% of height)")
    print(f"Width: {roi_width} ({roi_width/w*100:.1f}% of width)")
    print(f"Height: {roi_height} ({roi_height/h*100:.1f}% of height)")

# ----------------- GUI FUNCTIONS -----------------

def test_roi_extraction():
    """
    Test the ROI extraction on uploaded/captured image
    """
    global current_image
    if current_image is None:
        messagebox.showwarning("No Image", "Please select or capture an image first.")
        return
    
    # Extract using region-based method
    name, student_id = extract_name_and_id_from_region(current_image)
    
    # Show results
    result_window = tk.Toplevel(root)
    result_window.title("ROI Extraction Test")
    result_window.geometry("400x200")
    
    ttk.Label(result_window, text="Region-based Extraction Results:", font=("Helvetica", 12, "bold")).pack(pady=10)
    ttk.Label(result_window, text=f"Name: {name if name else 'Not found'}").pack(pady=5)
    ttk.Label(result_window, text=f"Student ID: {student_id if student_id else 'Not found'}").pack(pady=5)
    
    def close_test():
        result_window.destroy()
    
    ttk.Button(result_window, text="Close", command=close_test).pack(pady=20)  

def extract_name_and_id_from_region(image_pil):
    """
    Extract name and ID from the specific region (blue box area) of the ID card
    Based on the template coordinates where name and ID are located
    """
    # Apply auto-crop before OCR
    print("Applying auto-crop to improve OCR accuracy...")
    cropped_image_pil = auto_crop_id_card(image_pil)
    
    img_cv = cv2.cvtColor(np.array(cropped_image_pil), cv2.COLOR_RGB2BGR)
    h, w = img_cv.shape[:2]
    
    # Define the region of interest (ROI) based on template analysis
    # These coordinates are based on your template image layout
    # Adjust these values if needed based on your template_id.jpg dimensions
    roi_x = int(w * 0.52)  # Start from about 52% of width (right side)
    roi_y = int(h * 0.43)  # Start from about 43% of height
    roi_width = int(w * 0.46)  # Width of about 46% of total width
    roi_height = int(h * 0.25)  # Height of about 25% of total height
    
    # Extract the region of interest
    roi = img_cv[roi_y:roi_y+roi_height, roi_x:roi_x+roi_width]
    
    # Enhanced preprocessing for the ROI
    gray_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    
    # Apply morphological operations to clean up the text
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    cleaned = cv2.morphologyEx(gray_roi, cv2.MORPH_CLOSE, kernel)
    
    # Enhance contrast
    enhanced = cv2.convertScaleAbs(cleaned, alpha=1.4, beta=15)
    
    # Apply adaptive thresholding
    thresh = cv2.adaptiveThreshold(enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY, 11, 2)
    
    # Optional: Save the ROI for debugging (uncomment to see processed region)
    # cv2.imwrite("debug_roi.jpg", thresh)
    
    # Configure Tesseract for better accuracy
    custom_config = r'--oem 3 --psm 6 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 '
    
    # Extract text from ROI
    text = pytesseract.image_to_string(thresh, config=custom_config)
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    
    print(f"Extracted text from ROI: {lines}")
    
    name = ''
    student_id = ''
    
    # Process each line to find name and student ID
    for line in lines:
        line_cleaned = line.strip().upper()
        
        # Look for student ID pattern (e.g., 24WMR09148)
        id_match = re.search(r'\d{2}[A-Z]{3}\d{5}', line_cleaned)
        if id_match and not student_id:
            student_id = id_match.group()
            print(f"Found Student ID: {student_id}")
            continue
        
        # Look for name (should be alphabetic characters, multiple words)
        if (not re.search(r'\d', line_cleaned) and  # No digits in name
            len(line_cleaned.replace(" ", "")) > 3 and  # Minimum length
            re.match(r'^[A-Z\s]+$', line_cleaned) and  # Only letters and spaces
            len(line_cleaned.split()) >= 2 and  # At least 2 words (first name + last name)
            not any(keyword in line_cleaned for keyword in ['STUDENT', 'EXPIRY', 'DATE', 'TARUMT', 'MANAGEMENT', 'TECHNOLOGY'])):
            if not name:  # Take the first valid name found
                name = line_cleaned
                print(f"Found Name: {name}")
    
    # If we found an ID but no name, try to be more lenient with name detection
    if student_id and not name:
        for line in lines:
            line_cleaned = line.strip().upper()
            if (not re.search(r'\d', line_cleaned) and  # No digits
                len(line_cleaned.replace(" ", "")) > 2 and  # Minimum length
                re.match(r'^[A-Z\s]+$', line_cleaned) and  # Only letters and spaces
                line_cleaned != student_id):  # Not the same as student ID
                name = line_cleaned
                print(f"Found Name (lenient): {name}")
                break
    
    return name.strip(), student_id.strip()

    
def extract_name_and_id(image_pil):
    """
    Main extraction function that tries region-based extraction first,
    then falls back to full image extraction if needed
    """
    # Try region-based extraction first
    name, student_id = extract_name_and_id_from_region(image_pil)
    
    # If region-based extraction didn't work well, try full image extraction
    if not name or not student_id:
        print("Region-based extraction incomplete, trying full image...")
        name_full, id_full = extract_name_and_id_full_image(image_pil)
        
        # Use the best results from both methods
        if not name and name_full:
            name = name_full
        if not student_id and id_full:
            student_id = id_full
    
    return name.strip(), student_id.strip()

def extract_name_and_id_full_image(image_pil):
    """
    Fallback method: Extract from full image (original method)
    """
    print("Applying full image OCR extraction...")
    cropped_image_pil = auto_crop_id_card(image_pil)
    
    img_cv = cv2.cvtColor(np.array(cropped_image_pil), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    sharpened = cv2.addWeighted(gray, 1.5, blur, -0.5, 0)
    thresh = cv2.adaptiveThreshold(sharpened, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                   cv2.THRESH_BINARY, 15, 10)
    text = pytesseract.image_to_string(thresh)
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    name = ''
    student_id = ''

    for i, line in enumerate(lines):
        line_cleaned = line.replace("O", "0").replace("S", "5")
        match = re.search(r'\d{2}[A-Z0-9]{3}\d{5}', line_cleaned)
        if match:
            raw_id = match.group()
            student_id = raw_id.replace("0", "O", 1)
            for j in range(max(0, i - 3), i):
                candidate = lines[j]
                if (not re.search(r'\d', candidate)
                    and len(candidate.split()) >= 2
                    and candidate.isupper()
                    and not any(k in candidate for k in ['DATE', 'EXPIRY', 'STUDENT', 'TARUMT'])):
                    name = candidate
                    break
            break
    return name.strip(), student_id.strip()                   

# ----------------- QR CODE GENERATION -----------------
def generate_qr_code(student_name, student_id, email, folder_name):
    qr_data = f"Name: {student_name}\nID: {student_id}\nEmail: {email}"
    qr = qrcode.QRCode(version=1, box_size=10, border=5)
    qr.add_data(qr_data)
    qr.make(fit=True)
    img_qr = qr.make_image(fill_color="black", back_color="white")
    qr_path = os.path.join(folder_name, f"{student_name}.{student_id}_qr.png")
    img_qr.save(qr_path)
    return qr_path

# ----------------- EMAIL VALIDATION -----------------
def is_valid_email(email):
    if not re.match(r"[^@]+@[^@]+\.[^@]+", email):
        return False
    domain = email.split("@")[1]
    try:
        socket.gethostbyname(domain)
        return True
    except socket.error:
        return False

# ----------------- EMAIL SENDING -----------------
def send_email_with_qr(to_email, student_name, qr_path):
    try:
        subject = "🎓 Your Graduation QR Code"
        body = f"Dear {student_name},\n\nPlease find attached your unique QR code for the graduation ceremony.\nBring this QR code with you for scanning during the event.\n\nRegards,\nGraduation Committee"

        msg = MIMEMultipart()
        msg["From"] = SENDER_EMAIL
        msg["To"] = to_email
        msg["Subject"] = subject
        msg.attach(MIMEText(body, "plain"))

        with open(qr_path, "rb") as f:
            mime = MIMEBase("image", "png", filename=os.path.basename(qr_path))
            mime.add_header("Content-Disposition", "attachment", filename=os.path.basename(qr_path))
            mime.add_header("X-Attachment-Id", "0")
            mime.add_header("Content-ID", "<0>")
            mime.set_payload(f.read())
            encoders.encode_base64(mime)
            msg.attach(mime)

        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(SENDER_EMAIL, SENDER_PASSWORD)
        server.sendmail(SENDER_EMAIL, to_email, msg.as_string())
        server.quit()

        messagebox.showinfo("Success", f"QR Code sent to {to_email} successfully!")
        return True

    except Exception as e:
        messagebox.showerror("Email Error", f"Failed to send email: {e}")
        return False

# ----------------- EMAIL VERIFICATION -----------------
def send_verification_code(to_email):
    # 🔒 Check if email is locked BEFORE sending
    if to_email in failed_attempts:
        info = failed_attempts[to_email]
        if info["count"] >= 3 and datetime.now() < info["locked_until"]:
            messagebox.showerror(
                "Locked",
                f"Too many failed attempts for {to_email}. "
                f"Try again after {info['locked_until'].strftime('%H:%M:%S')}."
            )
            return False  # 🚫 don't send email

    try:
        code = str(random.randint(100000, 999999))
        verification_codes[to_email] = code

        subject = "🎓 Your Email Verification Code"
        body = f"Dear Student,\n\nYour verification code for convocation registration is: {code}\n\nDo not share this code with anyone."

        msg = MIMEMultipart()
        msg["From"] = SENDER_EMAIL
        msg["To"] = to_email
        msg["Subject"] = subject
        msg.attach(MIMEText(body, "plain"))

        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(SENDER_EMAIL, SENDER_PASSWORD)
        server.sendmail(SENDER_EMAIL, to_email, msg.as_string())
        server.quit()

        messagebox.showinfo("Verification Sent", f"A verification code has been sent to {to_email}.")
        return True

    except Exception as e:
        messagebox.showerror("Email Error", f"Failed to send verification code: {e}")
        return False

# ----------------- GUI FUNCTIONS -----------------

def display_image_pil(pil_img):
    global current_image
    current_image = pil_img
    img_resized = pil_img.resize((350, 250))
    img_tk = ImageTk.PhotoImage(img_resized)
    panel.config(image=img_tk)
    panel.image = img_tk
    confirm_btn.pack(pady=10)
    retake_btn.pack()
    upload_btn.pack_forget()
    capture_btn.pack_forget()

def upload_image():
    file_path = filedialog.askopenfilename(filetypes=[("Image files", "*.jpg *.jpeg *.png")])
    if file_path:
        pil_img = Image.open(file_path)
        display_image_pil(pil_img)

def take_picture():
    cap = cv2.VideoCapture(0)
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    rect_w, rect_h = 480, 300
    rect_x = (frame_width - rect_w) // 2
    rect_y = (frame_height - rect_h) // 2
    captured_frame = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        cv2.rectangle(frame, (rect_x, rect_y), (rect_x + rect_w, rect_y + rect_h), (0, 255, 0), 2)
        cv2.putText(frame, "Press SPACE to capture, ESC to quit", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)
        cv2.imshow("Capture ID Card", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == 32:
            captured_frame = frame[rect_y:rect_y + rect_h, rect_x:rect_x + rect_w]
            break
        elif key == 27:
            break
    cap.release()
    cv2.destroyAllWindows()
    if captured_frame is not None:
        pil_img = Image.fromarray(cv2.cvtColor(captured_frame, cv2.COLOR_BGR2RGB))
        display_image_pil(pil_img)

def confirm_image():
    global current_image
    if current_image is None:
        messagebox.showwarning("No Image", "Please select or capture an image first.")
        return

    name, student_id = extract_name_and_id(current_image)
    if not name and not student_id:
        messagebox.showwarning("OCR Failed", "Could not extract name or student ID.")
        return

    edit_window = tk.Toplevel(root)
    edit_window.title("Confirm Student Info")
    ttk.Label(edit_window, text="Please confirm or edit your information below:").grid(row=0, column=0, columnspan=2, pady=(10, 5))

    def force_uppercase_name(*args):
        current = name_var.get()
        name_var.set(current.upper())

    name_var = tk.StringVar()
    name_var.trace_add("write", force_uppercase_name)

    def force_uppercase_id(*args):
        current = id_var.get()
        id_var.set(current.upper())

    id_var = tk.StringVar()
    id_var.trace_add("write", force_uppercase_id)

    def force_lowercase_email(*args):
        current = email_var.get()
        email_var.set(current.lower())

    email_var = tk.StringVar()
    email_var.trace_add("write", force_lowercase_email)

    ttk.Label(edit_window, text="Name:").grid(row=1, column=0, padx=10, pady=5, sticky="e")
    name_entry = ttk.Entry(edit_window, width=40, textvariable=name_var)
    name_var.set(name)
    name_entry.grid(row=1, column=1, padx=10, pady=5)

    ttk.Label(edit_window, text="Student ID:").grid(row=2, column=0, padx=10, pady=5, sticky="e")
    id_entry = ttk.Entry(edit_window, width=40, textvariable=id_var)
    corrected_id = clean_student_id(student_id)
    id_var.set(corrected_id)
    id_entry.grid(row=2, column=1, padx=10, pady=5)

    ttk.Label(edit_window, text="School Email:").grid(row=3, column=0, padx=10, pady=5, sticky="e")
    email_entry = ttk.Entry(edit_window, width=40, textvariable=email_var)
    email_entry.grid(row=3, column=1, padx=10, pady=5)
    email_entry.focus_set()

    # ----------------- SAVE DATA -----------------
    def save_data():
        final_name = name_entry.get().strip().replace(" ", "_")
        final_id = id_entry.get().strip()
        email = email_entry.get().strip()

        if not final_name or not final_id or not email:
            messagebox.showerror("Error", "Name, Student ID, and Email are required.")
            return

        # Enforce school email
        school_email_pattern = r"^[a-z0-9]+-[a-z]{2}[0-9]{2}@student\.tarc\.edu\.my$"
        if not re.match(school_email_pattern, email.lower()):
            messagebox.showerror("Invalid Email", "Please enter a valid school email (e.g., tanyf-wm22@student.tarc.edu.my).")
            email_entry.focus_set()
            return

        # --- EMAIL VERIFICATION CHECK ---
        file_exists = os.path.exists("student_records.csv")
        if file_exists:
            with open("student_records.csv", "r", newline="") as fr:
                existing_emails = [row["Email"] for row in csv.DictReader(fr)]
            if email in existing_emails:
                messagebox.showerror("Email Already Used", 
                                    "This email has already been verified.\nPlease use a different school email.")
                email_entry.focus_set()  # Go back to email entry
                return  # Stop registration here
       
        # --- SEND VERIFICATION EMAIL ---
        if not send_verification_code(email):
            edit_window.destroy()      # Close confirm window
            retake_or_reselect()
            return  # stop if locked or failed to send
        
        # Ask for verification code
        def ask_verification_code():
            # check if this email is currently locked
            if email in failed_attempts and failed_attempts[email]["count"] >= 3:
                locked_until = failed_attempts[email]["locked_until"]
                if datetime.now() < locked_until:
                    messagebox.showerror(
                        "Locked",
                        f"Too many failed attempts. Try again after {locked_until.strftime('%H:%M:%S')}."
                    )
                    edit_window.destroy()      # ❌ close the confirm ID window
                    retake_or_reselect()       # ✅ back to homepage (Upload / Capture)
                    return  # stop completely

            code_window = tk.Toplevel(edit_window)
            code_window.title("Email Verification")
            ttk.Label(code_window, text=f"A verification code was sent to {email}").pack(pady=10)
            code_var = tk.StringVar()
            ttk.Entry(code_window, textvariable=code_var).pack(pady=5)

            def verify_code():
                user_code = code_var.get().strip()
                correct_code = verification_codes.get(email)

                if user_code == correct_code:
                    # Reset attempts on success
                    failed_attempts[email] = {"count": 0, "locked_until": datetime.min}
                    messagebox.showinfo("Verified", "Email verified successfully!")
                    code_window.destroy()
                    messagebox.showinfo("Next Step", "Now we will capture your face using the webcam.")
                    proceed_with_registration(final_name, final_id, email, edit_window)

                else:
                    # ❌ Wrong code
                    if email not in failed_attempts:
                        failed_attempts[email] = {"count": 0, "locked_until": datetime.min}

                    failed_attempts[email]["count"] += 1
                    remaining = 3 - failed_attempts[email]["count"]

                    if remaining > 0:
                        messagebox.showerror(
                            "Incorrect Code",
                            f"Wrong code. {remaining} attempt(s) left."
                        )
                    else:
                        # lock for 5 minutes
                        failed_attempts[email]["locked_until"] = datetime.now() + timedelta(minutes=5)
                        messagebox.showerror(
                            "Locked",
                            "Too many failed attempts. This email is locked for 5 minutes."
                        )
                        code_window.destroy()
                        edit_window.destroy()  # cancel registration too
                        retake_or_reselect()

            ttk.Button(code_window, text="Verify", command=verify_code).pack(pady=10)
            code_window.grab_set()
        ask_verification_code()
    ttk.Button(edit_window, text="Confirm & Save", command=save_data).grid(row=4, column=0, columnspan=2, pady=10)

def proceed_with_registration(final_name, final_id, email, edit_window):
    folder_name = os.path.join("StudentidFolder", f"{final_name}.{final_id}")
    os.makedirs(folder_name, exist_ok=True)
    img_path = os.path.join(folder_name, f"{final_name}.{final_id}.jpg")
    current_image.save(img_path)

    # CSV update
    file_exists = os.path.exists("student_records.csv")
    existing_ids = []
    if file_exists:
        with open("student_records.csv", "r", newline="") as fr:
            existing_ids = [row["Student ID"] for row in csv.DictReader(fr)]
    if final_id not in existing_ids:
        with open("student_records.csv", "a", newline="") as f:
            fieldnames = ["Name", "Student ID", "Email", "Image Path"]
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            if not file_exists or os.stat("student_records.csv").st_size == 0:
                writer.writeheader()
            writer.writerow({
                "Name": final_name.replace("_", " "),
                "Student ID": final_id,
                "Email": email,
                "Image Path": img_path
            })
    else:
        messagebox.showinfo("Info", f"Student {final_name} ({final_id}) already exists. Updated image path and email.")

    # --- FACE CAPTURE ---
    temp_folder_created = False
    frame_count = 0
    buffer_encodings = []
    capture_interval = 10
    timeout_seconds = 60
    start_time = time.time()
    cap = cv2.VideoCapture(0)
    from mtcnn import MTCNN
    detector = MTCNN()

    while True:
        ret, frame = cap.read()
        if not ret:
            continue

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = detector.detect_faces(rgb)
        h, w, _ = frame.shape
        center = (w // 2, h // 2)
        radius = 140
        segments = 40
        completed_ratio = len(buffer_encodings) / 7

        # Draw progress ring segments
        for i in range(segments):
            angle = 2 * math.pi * i / segments
            x1 = int(center[0] + radius * math.cos(angle))
            y1 = int(center[1] + radius * math.sin(angle))
            x2 = int(center[0] + (radius + 12) * math.cos(angle))
            y2 = int(center[1] + (radius + 12) * math.sin(angle))
            if i < math.floor(completed_ratio * segments):
                cv2.line(frame, (x1, y1), (x2, y2), (0, 200, 0), 2)
            else:
                cv2.line(frame, (x1, y1), (x2, y2), (180, 180, 180), 2)

        cv2.circle(frame, center, radius - 10, (255, 255, 255), 2)
        cv2.putText(frame, "Move your head slowly to complete the circle", (center[0] - 200, center[1] + radius + 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        # Face detection & encoding collection
        if len(results) == 1:
            if not temp_folder_created:
                os.makedirs(folder_name, exist_ok=True)
                temp_folder_created = True
                img_path = os.path.join(folder_name, f"{final_name}.{final_id}.jpg")
                current_image.save(img_path)
            if frame_count % capture_interval == 0:
                x, y, w_box, h_box = results[0]['box']
                top, right, bottom, left = y, x + w_box, y + h_box, x
                encodings = face_recognition.face_encodings(rgb, known_face_locations=[(top, right, bottom, left)])
                if encodings:
                    buffer_encodings.append(encodings[0])
                    if len(buffer_encodings) >= 7:
                        mean_encoding = np.mean(buffer_encodings, axis=0)
                        np.save(os.path.join(folder_name, "face_encoding.npy"), mean_encoding)
                        
                        # --- QR GENERATION & EMAIL AFTER FACE SUCCESS ---
                        qr_path = generate_qr_code(final_name, final_id, email, folder_name)
                        email_sent = send_email_with_qr(email, final_name, qr_path)
                        if email_sent:
                            messagebox.showinfo("Success", "Registration complete! You may now proceed to register the next student.")
                        break
        elif len(results) > 1:
            cv2.putText(frame, "Multiple faces detected", (60, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        frame_count += 1
        if time.time() - start_time > timeout_seconds:
            if temp_folder_created:
                try:
                    for f in os.listdir(folder_name):
                        os.remove(os.path.join(folder_name, f))
                    os.rmdir(folder_name)
                except:
                    pass
            messagebox.showwarning("Timeout", "No face detected in time. Registration failed. Please try again.")
            break

        cv2.imshow("Live Face Capture", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()
    edit_window.destroy()
    retake_or_reselect()

def retake_or_reselect():
    global current_image
    panel.config(image=None)
    panel.image = None
    current_image = None
    confirm_btn.pack_forget()
    retake_btn.pack_forget()
    upload_btn.pack(pady=10)
    capture_btn.pack(pady=5)

# ----------------- GUI SETUP -----------------
root = tk.Tk()
root.title("🎓 Convocation Registration System")
root.geometry("480x600")

welcome = ttk.Label(root, text="Welcome to Convocation Registration System!", font=("Helvetica", 14, "bold"))
welcome.pack(pady=10)

instruction = ttk.Label(root, text="Please register using your student ID card (upload or webcam)")
instruction.pack()

upload_btn = ttk.Button(root, text="Upload Student ID Card", command=upload_image)
upload_btn.pack(pady=10)

capture_btn = ttk.Button(root, text="Capture Student ID via Webcam", command=take_picture)
capture_btn.pack(pady=5)

panel = ttk.Label(root)
panel.pack(padx=10, pady=10)

confirm_btn = ttk.Button(root, text="Confirm ID Card", command=confirm_image)
retake_btn = ttk.Button(root, text="Retake / Select Another Image", command=retake_or_reselect)

root.mainloop()

Applying auto-crop to improve OCR accuracy...
No contours detected after pre-processing. Check lighting or card visibility.
Contour detection failed, trying template-based matching...
Found 548 good matches
Found 552 good matches
Found 545 good matches
Found 549 good matches
Extracted text from ROI: ['TAINGSFVAROTAIN', '4WMRO9148']
Region-based extraction incomplete, trying full image...
Applying full image OCR extraction...
No contours detected after pre-processing. Check lighting or card visibility.
Contour detection failed, trying template-based matching...
Found 548 good matches
Found 550 good matches
Found 546 good matches
Found 546 good matches


Admin Assign CGPA to Student

In [5]:
import tkinter as tk
from tkinter import ttk, messagebox
import csv
import os
import re

# ----------------- Admin UI for CGPA and Award Management -----------------
def load_student_records():
    """Load student records from CSV into a list of dictionaries."""
    student_records = []
    if os.path.exists("student_records.csv"):
        try:
            with open("student_records.csv", "r", newline="") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    student_records.append({
                        "Name": row.get("Name", ""),
                        "Student ID": row.get("Student ID", ""),
                        "Email": row.get("Email", ""),
                        "Image Path": row.get("Image Path", ""),
                        "CGPA": row.get("CGPA", ""),
                        "Award_Category": row.get("Award_Category", ""),
                        "Category_Sequence": row.get("Category_Sequence", "")
                    })
        except Exception as e:
            print(f"Error loading student_records.csv: {e}")
            messagebox.showerror("Error", f"Failed to load student_records.csv: {e}")
    return student_records

def assign_award_category(cgpa):
    """Assign award category based on CGPA."""
    try:
        cgpa = float(cgpa)
        if cgpa >= 3.75:
            return "DISTINCTION"
        elif cgpa >= 2.75:
            return "MERIT"
        else:
            return ""
    except ValueError:
        return ""

def update_category_sequences(students):
    """Assign sequence numbers within each award category."""
    category_groups = {"DISTINCTION": [], "MERIT": []}
    for student in students:
        category = student["Award_Category"]
        if category in category_groups:
            category_groups[category].append(student)

    for category in category_groups:
        for index, student in enumerate(sorted(category_groups[category], key=lambda x: float(x["CGPA"] or 0), reverse=True)):
            student["Category_Sequence"] = str(index + 1)

def save_student_records(students):
    """Save updated student records to CSV."""
    fieldnames = ["Name", "Student ID", "Email", "Image Path", "CGPA", "Award_Category", "Category_Sequence"]
    try:
        with open("student_records.csv", "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for student in students:
                writer.writerow(student)
        print("Successfully saved to student_records.csv")
    except Exception as e:
        print(f"Error saving to student_records.csv: {e}")
        messagebox.showerror("Error", f"Failed to save to student_records.csv: {e}")
        raise

def create_admin_ui():
    root = tk.Tk()
    root.title("Admin CGPA and Award Manager")
    root.geometry("800x600")

    # Instructions
    ttk.Label(root, text="Admin: Manage CGPA and Award Categories", font=("Helvetica", 14, "bold")).pack(pady=10)
    ttk.Label(root, text="Enter CGPA (exactly 4 decimal places, e.g., 3.1234) for each student to save.").pack(pady=5)

    # Load student records
    students = load_student_records()
    if not students:
        messagebox.showwarning("No Records", "No student records found in student_records.csv.")
        root.destroy()
        return

    # Treeview for displaying and editing student data
    columns = ("Name", "Student ID", "Email", "CGPA", "Award_Category", "Category_Sequence")
    tree = ttk.Treeview(root, columns=columns, show="headings", height=15)
    tree.pack(padx=10, pady=10, fill="both", expand=True)

    # Set column headings
    tree.heading("Name", text="Name")
    tree.heading("Student ID", text="Student ID")
    tree.heading("Email", text="Email")
    tree.heading("CGPA", text="CGPA")
    tree.heading("Award_Category", text="Award Category")
    tree.heading("Category_Sequence", text="Category Sequence")

    # Set column widths
    tree.column("Name", width=150)
    tree.column("Student ID", width=100)
    tree.column("Email", width=200)
    tree.column("CGPA", width=80)
    tree.column("Award_Category", width=120)
    tree.column("Category_Sequence", width=120)

    # Populate treeview with student data
    for student in students:
        tree.insert("", "end", values=(
            student["Name"],
            student["Student ID"],
            student["Email"],
            student["CGPA"],
            student["Award_Category"],
            student["Category_Sequence"]
        ))

    # CGPA entry and update button
    def update_cgpa():
        selected_item = tree.selection()
        if not selected_item:
            messagebox.showwarning("No Selection", "Please select a student to update CGPA.")
            return

        cgpa_window = tk.Toplevel(root)
        cgpa_window.title("Update CGPA")
        cgpa_window.geometry("300x150")
        cgpa_window.transient(root)
        cgpa_window.grab_set()

        ttk.Label(cgpa_window, text="Enter CGPA (0.0000 - 4.0000):").pack(pady=10)
        cgpa_var = tk.StringVar()
        cgpa_entry = ttk.Entry(cgpa_window, textvariable=cgpa_var)
        cgpa_entry.pack(pady=5)
        cgpa_entry.focus_set()

        def save_cgpa():
            try:
                cgpa = cgpa_var.get().strip()
                print(f"Attempting to save CGPA: {cgpa}")
                if not re.match(r"^\d\.\d{4}$", cgpa):
                    messagebox.showerror("Invalid CGPA", "CGPA must have exactly 4 decimal places (e.g., 3.1234).", parent=cgpa_window)
                    print("Validation failed: Incorrect CGPA format")
                    return
                cgpa_float = float(cgpa)
                if not 0.0000 <= cgpa_float <= 4.0000:
                    messagebox.showerror("Invalid CGPA", "CGPA must be between 0.0000 and 4.0000.", parent=cgpa_window)
                    print("Validation failed: CGPA out of range")
                    return
                student_name = tree.item(selected_item)["values"][0]
                print(f"Selected student: {student_name}")
                selected_id = tree.item(selected_item)["values"][1] 
                print(f"Updating CGPA for Student ID: {selected_id}")
                for student in students:
                    if student["Student ID"] == selected_id:
                        student["CGPA"] = cgpa
                        student["Award_Category"] = assign_award_category(cgpa)
                        break
                print("Updating category sequences")
                update_category_sequences(students)
                print("Saving to CSV")
                save_student_records(students)
                # Update treeview
                print("Refreshing Treeview")
                for item in tree.get_children():
                    tree.delete(item)
                for student in students:
                    tree.insert("", "end", values=(
                        student["Name"],
                        student["Student ID"],
                        student["Email"],
                        student["CGPA"],
                        student["Award_Category"],
                        student["Category_Sequence"]
                    ))
                messagebox.showinfo("Success", f"CGPA updated for {student_name} and saved to CSV.")
                print("CGPA update successful, closing CGPA window")
                cgpa_window.grab_release()  
                cgpa_window.destroy() 
            except Exception as e:
                messagebox.showerror("Error", f"Failed to save CGPA: {e}", parent=cgpa_window)
                print(f"Error in save_cgpa: {e}")
                cgpa_window.grab_release()  
                cgpa_window.destroy()  

        ttk.Button(cgpa_window, text="Save CGPA", command=save_cgpa).pack(pady=10)

    # Buttons
    ttk.Button(root, text="Update Selected Student's CGPA", command=update_cgpa).pack(pady=5)
    ttk.Button(root, text="Exit", command=root.destroy).pack(pady=5)

    root.mainloop()

if __name__ == "__main__":
    create_admin_ui()

Attempting to save CGPA: 3.8888
Selected student: CHANG KAR YES
Updating CGPA for Student ID: 24WMR09222
Updating category sequences
Saving to CSV
Successfully saved to student_records.csv
Refreshing Treeview
CGPA update successful, closing CGPA window


DURING CEREMONY QR + FACE

In [6]:
import cv2
import numpy as np
import face_recognition
import csv
from pyzbar import pyzbar
import tkinter as tk
from tkinter import messagebox, ttk
import time
import os
import pyttsx3

# ----------------- Helper for popups -----------------
def show_popup(title, message, parent):
    popup = tk.Toplevel(parent)
    popup.title(title)
    popup.geometry("300x100")
    ttk.Label(popup, text=message, wraplength=250).pack(pady=10)
    ttk.Button(popup, text="OK", command=popup.destroy).pack(pady=5)
    popup.transient(parent)
    popup.grab_set()
    parent.wait_window(popup)

# ----------------- Select Award Category -----------------
def select_award_category(parent):
    category_window = tk.Toplevel(parent)
    category_window.title("Select Award Category")
    category_window.geometry("300x150")
    
    ttk.Label(category_window, text="Select Award Category for Ceremony:", font=("Helvetica", 12)).pack(pady=10)
    
    categories = ["DISTINCTION", "MERIT"]
    category_var = tk.StringVar()
    category_dropdown = ttk.Combobox(category_window, textvariable=category_var, values=categories, state="readonly")
    category_dropdown.pack(pady=10)
    category_dropdown.set(categories[0])  # Default to first category
    
    def confirm_selection():
        selected = category_var.get()
        if selected:
            category_window.destroy()
            parent.selected_category = selected  # Store selected category
            parent.category_selected = True  # Signal category selection
            category_window.quit()
        else:
            messagebox.showerror("Error", "Please select a category.", parent=category_window)
    
    ttk.Button(category_window, text="Confirm", command=confirm_selection).pack(pady=10)
    category_window.transient(parent)
    category_window.grab_set()
    category_window.mainloop()
    return category_var.get()

# ----------------- Load students for selected category -----------------
def load_students_for_category(selected_category, student_encodings, students_list, attendance_marked_students):
    student_encodings.clear()
    students_list.clear()
    attendance_marked_students.clear()
    
    student_records_file = "student_records.csv"
    if not os.path.exists(student_records_file):
        print(f"Error: {student_records_file} does not exist.")
        return
    
    with open(student_records_file, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get('Award_Category') != selected_category:
                continue
            name = row.get('Name', '').replace(" ", "_")
            sid = row.get('Student ID', '')
            image_path = row.get('Image Path', '')
            sequence = row.get('Category_Sequence', '')
            folder_key = f"{name}.{sid}"
            encoding_path = os.path.join("StudentidFolder", folder_key, "face_encoding.npy")
            
            print(f"Processing student: {folder_key}, Award_Category: {row.get('Award_Category')}")
            
            if os.path.exists(encoding_path):
                try:
                    encoding = np.load(encoding_path)
                    student_encodings[folder_key] = encoding
                    students_list.append({'name': name, 'sid': sid, 'key': folder_key, 'sequence': sequence})
                    print(f"Loaded face encoding for {folder_key}")
                except Exception as e:
                    print(f"Failed to load face encoding for {folder_key}: {e}")
            else:
                print(f"No face encoding file found for {folder_key} at {encoding_path}")
            
            # Check if image path exists (for debugging, not used for encoding)
            if not os.path.exists(image_path):
                print(f"Image not found for {folder_key} at {image_path}")
    
    # Load marked students for this category from attendance.csv
    attendance_file = "attendance.csv"
    if os.path.exists(attendance_file):
        with open(attendance_file, "r") as f:
            next(f)  # skip header
            for line in f:
                cols = line.strip().split(",")
                if len(cols) >= 3 and cols[2] == "Present":
                    folder_key = f"{cols[0].replace(' ', '_')}.{cols[1]}"
                    if folder_key in student_encodings:
                        attendance_marked_students.add(folder_key)
                        print(f"Marked as present: {folder_key}")
    
    print(f"Loaded {len(students_list)} students for category {selected_category}")

# ----------------- TTS --------------------------
def talk_student_name(student_name, sequence, category):
    try:
        engine = pyttsx3.init()
        engine.setProperty('rate', 150)
        engine.setProperty('volume', 0.8)
        announcement = f"{student_name}, {category}"
        print(f"Speaking: {announcement}")
        engine.say(announcement)
        engine.runAndWait()
    except Exception as e:
        print(f"TTS error: {e}")
    finally:
        if 'engine' in locals():
            engine.stop()

# ----------------- Main Program -----------------
def main():
    student_records_file = "student_records.csv"
    attendance_file = "attendance.csv"
    student_encodings = {}
    students_list = []
    attendance_marked_students = set()

    # Initialize Tkinter root
    root = tk.Tk()
    root.title("Student Check-in System")
    root.geometry("300x200")
    root.selected_category = None
    root.category_selected = False

    # Select initial category
    select_award_category(root)
    if not root.category_selected:
        print("No category selected. Exiting.")
        root.destroy()
        return

    selected_category = root.selected_category
    category_label = ttk.Label(root, text=f"Current Category: {selected_category}", font=("Helvetica", 12))
    category_label.pack(pady=10)

    # Load students for initial category
    load_students_for_category(selected_category, student_encodings, students_list, attendance_marked_students)

    # Setup attendance file
    if not os.path.exists(attendance_file):
        with open(attendance_file, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["Name", "Student ID", "Status"])
            for student in students_list:
                writer.writerow([student['name'].replace("_", " "), student['sid'], "Absent"])
    else:
        print("📄 Using existing attendance.csv (will keep previous marks).")

    # Check if students were loaded
    if not student_encodings:
        show_popup("Error", f"No registered students found for {selected_category}. Exiting.", root)
        print(f"=== No registered student encodings found for {selected_category}. ===")
        root.destroy()
        return

    # Webcam setup
    cap = cv2.VideoCapture(0)
    font = cv2.FONT_HERSHEY_SIMPLEX

    qr_verified_student = None
    current_step = "QR Scan"
    face_match_start_time = None
    countdown_time = 3  # seconds needed to confirm face
    print(f"📷 Webcam started. Please scan your QR code inside the box for {selected_category} category.")

    def end_session():
        nonlocal selected_category, qr_verified_student, current_step, face_match_start_time, cap
        cap.release()
        cv2.destroyAllWindows()
        root.category_selected = False
        select_award_category(root)
        if not root.category_selected:
            print("No category selected. Exiting.")
            root.destroy()
            root.quit()
            return
        selected_category = root.selected_category
        category_label.config(text=f"Current Category: {selected_category}")
        load_students_for_category(selected_category, student_encodings, students_list, attendance_marked_students)
        if not student_encodings:
            show_popup("Error", f"No registered students found for {selected_category}. Exiting.", root)
            print(f"=== No registered student encodings found for {selected_category}. ===")
            root.destroy()
            root.quit()
            return
        qr_verified_student = None
        current_step = "QR Scan"
        face_match_start_time = None
        print(f"📷 Webcam restarted for {selected_category} category.")
        cap = cv2.VideoCapture(0)

    while True:
        ret, frame = cap.read()
        if not ret:
            continue

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        h, w, _ = frame.shape
        y_offset = 30

        # --- Define boxes ---
        qr_box_h = h // 3
        qr_box_w = w // 3
        qr_box_top = (h - qr_box_h) // 2
        qr_box_bottom = qr_box_top + qr_box_h
        qr_box_left = (w - qr_box_w) // 2
        qr_box_right = qr_box_left + qr_box_w

        face_box_h = h * 2 // 3
        face_box_w = w * 2 // 3
        face_box_top = (h - face_box_h) // 2
        face_box_bottom = face_box_top + face_box_h
        face_box_left = (w - face_box_w) // 2
        face_box_right = face_box_left + face_box_w

        # Draw boxes
        if current_step == "QR Scan":
            cv2.rectangle(frame, (qr_box_left, qr_box_top), (qr_box_right, qr_box_bottom), (0, 255, 255), 2)
        else:
            cv2.rectangle(frame, (face_box_left, face_box_top), (face_box_right, face_box_bottom), (255, 255, 0), 2)

        # ===== QR SCANNING PHASE =====
        if current_step == "QR Scan" and qr_verified_student is None:
            cv2.putText(frame, f"Align QR inside the box ({selected_category})", (qr_box_left, qr_box_top - 10),
                        font, 0.6, (0, 255, 255), 2)

            decoded_qrs = pyzbar.decode(frame)
            for qr in decoded_qrs:
                (x, y, qr_w, qr_h) = qr.rect
                qr_center_x = x + qr_w // 2
                qr_center_y = y + qr_h // 2

                if qr_box_left < qr_center_x < qr_box_right and qr_box_top < qr_center_y < qr_box_bottom:
                    qr_data = qr.data.decode("utf-8")
                    lines = qr_data.split("\n")
                    try:
                        name_line = [l for l in lines if "Name:" in l][0]
                        id_line = [l for l in lines if "ID:" in l][0]
                        student_name = name_line.split(":")[1].strip().replace(" ", "_")
                        student_id = id_line.split(":")[1].strip()
                        folder_key = f"{student_name}.{student_id}"

                        if folder_key in student_encodings:
                            if folder_key in attendance_marked_students:
                                show_popup("Already Taken",
                                           f"Attendance already marked for {student_name.replace('_',' ')} ({student_id})", root)
                            else:
                                qr_verified_student = folder_key
                                current_step = "Face Scan"
                                show_popup("QR Verified", f"QR Verified!\n{student_name.replace('_',' ')} ({student_id})", root)
                        else:
                            show_popup("Invalid QR Code", f"This student is not in the {selected_category} category!", root)
                    except:
                        show_popup("Invalid QR Code", "QR Code format is not valid!", root)

        # ===== FACE RECOGNITION PHASE =====
        elif current_step == "Face Scan" and qr_verified_student:
            face_locations = face_recognition.face_locations(rgb)
            face_encodings = face_recognition.face_encodings(rgb, face_locations)

            matched = False
            for (top, right, bottom, left), live_encoding in zip(face_locations, face_encodings):
                face_center_x = (left + right) // 2
                face_center_y = (top + bottom) // 2
                if not (face_box_left < face_center_x < face_box_right and face_box_top < face_center_y < face_box_bottom):
                    cv2.putText(frame, "Keep face inside the box!", (10, h - 20), font, 0.7, (0, 0, 255), 2)
                    continue

                db_encoding = student_encodings[qr_verified_student]
                distance = face_recognition.face_distance([db_encoding], live_encoding)[0]

                if distance < 0.6:
                    cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 3)
                    cv2.putText(frame, "Face Match", (left, top - 10), font, 0.7, (0, 255, 0), 2)
                    matched = True
                else:
                    cv2.rectangle(frame, (left, top), (right, bottom), (0, 0, 255), 3)
                    cv2.putText(frame, "Not matching!", (left, top - 10), font, 0.6, (0, 0, 255), 2)

            # Handle countdown if matched
            if matched:
                if face_match_start_time is None:
                    face_match_start_time = time.time()
                elapsed = int(time.time() - face_match_start_time)
                remaining = countdown_time - elapsed

                if remaining > 0:
                    cv2.putText(frame, f"Hold still... {remaining}", (50, h - 50), font, 0.8, (0, 255, 0), 2)
                else:
                    # Update attendance to Present
                    name, sid = qr_verified_student.split(".", 1)
                    sequence = next(s['sequence'] for s in students_list if s['key'] == qr_verified_student)
                    rows = []
                    with open(attendance_file, "r") as f:
                        reader = csv.reader(f)
                        rows = list(reader)
                    with open(attendance_file, "w", newline="") as f:
                        writer = csv.writer(f)
                        for row in rows:
                            if len(row) >= 3 and row[0].replace(" ", "_") == name and row[1] == sid:
                                row[2] = "Present"
                            writer.writerow(row)

                    attendance_marked_students.add(qr_verified_student)
                    show_popup("Attendance Marked", f"Attendance marked for {name.replace('_',' ')} ({sid})", root)
                    talk_student_name(name.replace('_', ' '), sequence, selected_category)

                    qr_verified_student = None
                    current_step = "QR Scan"
                    face_match_start_time = None
            else:
                face_match_start_time = None

        # ===== DISPLAY STATUS =====
        status_text = f"Step: {current_step} ({selected_category})"
        cv2.putText(frame, status_text, (10, y_offset), font, 0.7, (0, 0, 0), 2)

        if qr_verified_student:
            name, sid = qr_verified_student.split(".", 1)
            sequence = next(s['sequence'] for s in students_list if s['key'] == qr_verified_student)
            cv2.putText(frame, f"{name.replace('_',' ')} - {sid}",
                        (10, y_offset + 30), font, 0.7, (0, 255, 0), 2)

        cv2.putText(frame, "Press Q to end session, ESC to quit", (10, y_offset + 70), font, 0.6, (0, 0, 0), 2)
        cv2.imshow("Student Check-in System", frame)

        # Handle keypresses
        key = cv2.waitKey(1) & 0xFF
        if key == 27:  # ESC to quit
            break
        elif key == ord('q') or key == ord('Q'):  # Q to end session
            end_session()

    cap.release()
    cv2.destroyAllWindows()
    root.destroy()

if __name__ == "__main__":
    main()

Processing student: CHANG_KAR_YAN.24WMR09148, Award_Category: DISTINCTION
Loaded face encoding for CHANG_KAR_YAN.24WMR09148
Processing student: CHANG_KAR_YES.24WMR09222, Award_Category: DISTINCTION
Loaded face encoding for CHANG_KAR_YES.24WMR09222
Marked as present: CHANG_KAR_YAN.24WMR09148
Loaded 2 students for category DISTINCTION
📄 Using existing attendance.csv (will keep previous marks).
📷 Webcam started. Please scan your QR code inside the box for DISTINCTION category.
Speaking: CHANG KAR YES, DISTINCTION
